In [1]:
# 1. Mount Google Drive (results, figures and checkpoints are saved under MyDrive/shaaer_v2)
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/shaaer_code/shaaer', exist_ok=True)
open('/content/shaaer_code/shaaer/__init__.py', 'w').close()
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


In [2]:
%%writefile /content/shaaer_code/shaaer/tok.py
"""BPE tokeniser for Arabic poetry (from-scratch, Sennrich-style).

Same algorithm as the original notebook: diacritics are glued to the letter
they follow before BPE (a no-op on tashkeel-stripped text), merges never cross
a space, and the tokeniser is fitted on the training split only.

Changes from the notebook: encoding caches per word (the corpus has ~30M word
occurrences but far fewer distinct words), and the tokeniser saves to JSON.
"""
import json
import re
from collections import Counter

COMBINING = "ؐ-ًؚ-ٰٟۖ-ۭ"
CLUSTER_RE = re.compile(rf"[^{COMBINING}][{COMBINING}]*|[{COMBINING}]+")


def to_clusters(s):
    return CLUSTER_RE.findall(s)


class ArabicBPE:
    def __init__(self, vocab, merges):
        self.vocab = list(vocab)
        self.merges = [tuple(m) for m in merges]
        self.stoi = {t: i for i, t in enumerate(self.vocab)}
        self.ranks = {p: i for i, p in enumerate(self.merges)}
        self._cache = {}

    @property
    def vocab_size(self):
        return len(self.vocab)

    @classmethod
    def train(cls, text, vocab_size=512, verbose=False):
        words = Counter(text.split())
        splits = {w: tuple(to_clusters(w)) for w in words}
        alpha = sorted({c for sp in splits.values() for c in sp})
        vocab = ["\n", " "] + [c for c in alpha if c not in ("\n", " ")]
        merges = []
        pairs, where = Counter(), {}
        for w, f in words.items():
            sp = splits[w]
            for i in range(len(sp) - 1):
                p = (sp[i], sp[i + 1])
                pairs[p] += f
                where.setdefault(p, set()).add(w)
        while len(vocab) < vocab_size and pairs:
            best, cnt = max(pairs.items(), key=lambda kv: kv[1])
            if cnt < 2:
                break
            a, b = best
            new = a + b
            merges.append(best)
            vocab.append(new)
            for w in list(where.get(best, ())):
                sp, f = splits[w], words[w]
                for i in range(len(sp) - 1):
                    p = (sp[i], sp[i + 1])
                    pairs[p] -= f
                    if pairs[p] <= 0:
                        pairs.pop(p, None)
                        where.pop(p, None)
                    elif p in where:
                        where[p].discard(w)
                out, i = [], 0
                while i < len(sp):
                    if i < len(sp) - 1 and sp[i] == a and sp[i + 1] == b:
                        out.append(new)
                        i += 2
                    else:
                        out.append(sp[i])
                        i += 1
                sp = tuple(out)
                splits[w] = sp
                for i in range(len(sp) - 1):
                    p = (sp[i], sp[i + 1])
                    pairs[p] += f
                    where.setdefault(p, set()).add(w)
            pairs.pop(best, None)
            where.pop(best, None)
            if verbose and len(merges) % 100 == 0:
                print(f"  {len(merges)} merges", flush=True)
        return cls(vocab, merges)

    def _word(self, w):
        hit = self._cache.get(w)
        if hit is not None:
            return hit
        sp = to_clusters(w)
        while len(sp) > 1:
            best, br = None, None
            for i in range(len(sp) - 1):
                r = self.ranks.get((sp[i], sp[i + 1]))
                if r is not None and (br is None or r < br):
                    best, br = i, r
            if best is None:
                break
            sp = sp[:best] + [sp[best] + sp[best + 1]] + sp[best + 2:]
        ids = []
        for tok in sp:
            if tok in self.stoi:
                ids.append(self.stoi[tok])
            else:  # unseen cluster: fall back to codepoints, drop unknowns
                ids.extend(self.stoi[ch] for ch in tok if ch in self.stoi)
        self._cache[w] = ids
        return ids

    def encode(self, text):
        nl, sp = self.stoi["\n"], self.stoi[" "]
        ids = []
        for j, line in enumerate(text.split("\n")):
            if j:
                ids.append(nl)
            for k, w in enumerate(line.split(" ")):
                if k:
                    ids.append(sp)
                if w:
                    ids.extend(self._word(w))
        return ids

    def decode(self, ids):
        return "".join(self.vocab[i] for i in ids)

    def token_char_lengths(self):
        return [len(t) for t in self.vocab]

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab": self.vocab, "merges": self.merges}, f, ensure_ascii=False)

    @classmethod
    def load(cls, path):
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        return cls(d["vocab"], d["merges"])


Writing /content/shaaer_code/shaaer/tok.py


In [11]:
%%writefile /content/shaaer_code/shaaer/data.py
"""Corpus build, de-duplication, poem-level split, tokenisation, caching.

Changes from the original pipeline:
  * exact-duplicate poems are removed (Ashaar is scraped from several sites);
  * the split is a seeded shuffle of POEMS (95/5), not "the last 10% in dataset
    order", so validation is an exchangeable sample of the same distribution;
  * val/train bait overlap is measured and reported;
  * token ids are cached as uint16 arrays, with a per-token character length
    table so bits-per-character is exact on any subset of tokens.
"""
import bz2
import gzip
import hashlib
import json
import lzma
import math
import os
import random
import re
import time
import unicodedata

import numpy as np

from .tok import ArabicBPE

TASHKEEL = re.compile(r"[ؐ-ًؚ-ٰٟۖ-ۭ]")
TATWEEL = "ـ"
ARABIC = "ء-غف-يٱ"
ALLOWED = re.compile(rf"[^{ARABIC}ً-ْٰ ]")


def clean(s):
    s = unicodedata.normalize("NFKC", s).replace(TATWEEL, "")
    s = TASHKEEL.sub("", s)
    s = re.sub("[إأآٱ]", "ا", s)
    s = s.replace("ى", "ي").replace("ة", "ه")
    s = ALLOWED.sub(" ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalize_prompt(s):
    """Prompts must go through the same normalisation as training text."""
    s = unicodedata.normalize("NFKC", s).replace(TATWEEL, "")
    s = TASHKEEL.sub("", s)
    s = re.sub("[إأآٱ]", "ا", s)
    return s.replace("ى", "ي").replace("ة", "ه")


def short_meter(m):
    return (m or "").replace("بحر ", "").strip() or "غير معروف"


def short_theme(t):
    return (t or "").replace("قصيدة ", "").strip() or "عامه"


def poems_from_ashaar(ds, min_baits=3, max_baits=60):
    """Yield (header, [bait, ...]) per poem. Hemistichs come from the dataset's
    own sadr/ajuz list, so a word split across ' # ' is the source's own
    segmentation (tadwir), not something this code introduces."""
    stats = dict(seen=0, odd=0, too_short=0, truncated=0)
    out = []
    for rec in ds:
        stats["seen"] += 1
        hemis = rec.get("poem verses") or []
        if len(hemis) % 2:
            stats["odd"] += 1
            hemis = hemis[:-1]
        baits = []
        for i in range(0, len(hemis), 2):
            a, b = clean(hemis[i]), clean(hemis[i + 1])
            if len(a.split()) < 2 or len(b.split()) < 2:
                continue
            baits.append(f"{a} # {b}")
        if len(baits) < min_baits:
            stats["too_short"] += 1
            continue
        if len(baits) > max_baits:
            baits = baits[:max_baits]
            stats["truncated"] += 1
        header = f"[{short_meter(rec.get('poem meter'))}|{short_theme(rec.get('poem theme'))}]"
        out.append((header, baits))
    return out, stats


def poem_text(header, baits):
    return header + "\n" + "\n".join(baits)


def compressor_baselines(text):
    raw = text.encode("utf-8")
    n = len(text)
    rows = {"uniform": math.log2(len(set(text)))}
    for name, mod in (("gzip", gzip), ("bzip2", bz2), ("lzma", lzma)):
        rows[name] = len(mod.compress(raw)) * 8 / n
    return rows


def build(out_dir, poems, vocab_size=512, val_frac=0.05, seed=0, log=print):
    """poems: list of (header, baits). Writes train.bin/val.bin (uint16),
    tokenizer.json, meta.json, train_poems.txt, val_poems.txt."""
    os.makedirs(out_dir, exist_ok=True)
    meta_path = os.path.join(out_dir, "meta.json")
    if os.path.exists(meta_path):
        log(f"data already built in {out_dir}")
        return json.load(open(meta_path))

    t0 = time.time()
    # exact-duplicate poems (same baits, ignoring header)
    seen, uniq = set(), []
    for h, b in poems:
        key = hashlib.md5("\n".join(b).encode()).hexdigest()
        if key in seen:
            continue
        seen.add(key)
        uniq.append((h, b))
    n_dup = len(poems) - len(uniq)
    log(f"poems: {len(poems):,}  exact duplicates removed: {n_dup:,}")

    rng = random.Random(seed)
    rng.shuffle(uniq)
    n_val = int(len(uniq) * val_frac)
    val, train = uniq[:n_val], uniq[n_val:]

    train_baits = set(b for _, bs in train for b in bs)
    val_baits = [b for _, bs in val for b in bs]
    overlap_raw = sum(b in train_baits for b in val_baits) / max(1, len(val_baits))
    log(f"val baits also present verbatim in train: {100*overlap_raw:.2f}% (before cleaning)")
    # near-duplicates: a val poem sharing any line with train goes to train
    moved = 0
    for _ in range(5):
        keep, move = [], []
        for p in val:
            (move if any(b in train_baits for b in p[1]) else keep).append(p)
        if not move:
            break
        moved += len(move)
        train += move
        for _, bs in move:
            train_baits.update(bs)
        val = keep
    val_baits = [b for _, bs in val for b in bs]
    overlap = sum(b in train_baits for b in val_baits) / max(1, len(val_baits))
    log(f"moved {moved:,} near-duplicate poems from val to train; "
        f"val overlap now {100*overlap:.2f}% ({len(val):,} val poems)")

    train_text = "\n\n".join(poem_text(h, b) for h, b in train) + "\n"
    val_text = "\n\n".join(poem_text(h, b) for h, b in val) + "\n"
    open(os.path.join(out_dir, "train_poems.txt"), "w", encoding="utf-8").write(train_text)
    open(os.path.join(out_dir, "val_poems.txt"), "w", encoding="utf-8").write(val_text)

    log(f"training BPE-{vocab_size} on {len(train_text):,} chars ...")
    t1 = time.time()
    tok = ArabicBPE.train(train_text, vocab_size)
    log(f"  {len(tok.merges)} merges in {time.time()-t1:.0f}s")
    tok.save(os.path.join(out_dir, "tokenizer.json"))

    probe = val_text[:20000]
    assert tok.decode(tok.encode(probe)) == probe, "tokeniser round trip failed"

    ids = {}
    for name, txt in (("train", train_text), ("val", val_text)):
        t1 = time.time()
        a = np.array(tok.encode(txt), dtype=np.uint16)
        a.tofile(os.path.join(out_dir, f"{name}.bin"))
        ids[name] = a
        log(f"  {name}: {len(txt):,} chars -> {len(a):,} tokens "
            f"({len(txt)/len(a):.3f} chars/token, {time.time()-t1:.0f}s)")

    base = compressor_baselines(val_text)
    log("compressors on val text (bits/char): " +
        ", ".join(f"{k} {v:.3f}" for k, v in base.items()))

    headers = {}
    for h, _ in train:
        headers[h] = headers.get(h, 0) + 1
    top_headers = sorted(headers.items(), key=lambda kv: -kv[1])[:20]

    meta = dict(
        vocab_size=tok.vocab_size, n_merges=len(tok.merges),
        n_poems_raw=len(poems), n_duplicates=n_dup,
        n_train_poems=len(train), n_val_poems=len(val),
        train_chars=len(train_text), val_chars=len(val_text),
        train_tokens=int(len(ids["train"])), val_tokens=int(len(ids["val"])),
        val_bait_overlap=overlap, val_bait_overlap_raw=overlap_raw,
        n_moved_to_train=moved, compressors_val=base,
        top_headers=top_headers, seed=seed, val_frac=val_frac,
        build_seconds=time.time() - t0,
    )
    json.dump(meta, open(meta_path, "w"), ensure_ascii=False, indent=1)
    return meta


def load(out_dir):
    meta = json.load(open(os.path.join(out_dir, "meta.json")))
    tok = ArabicBPE.load(os.path.join(out_dir, "tokenizer.json"))
    train = np.fromfile(os.path.join(out_dir, "train.bin"), dtype=np.uint16)
    val = np.fromfile(os.path.join(out_dir, "val.bin"), dtype=np.uint16)
    return meta, tok, train, val


Overwriting /content/shaaer_code/shaaer/data.py


In [4]:
%%writefile /content/shaaer_code/shaaer/model.py
"""Configurable decoder-only transformer.

Attention variants
  attn="mha"       standard multi-head attention. Each head has its own W_Q, W_K
                   (d -> qk_dim) and W_V (d -> head_dim). Scores x_t W_Q W_K^T x_s
                   use a bilinear form of rank <= qk_dim.
  attn="bilinear"  W_Q W_K^T replaced by ONE learnable d x d matrix M_h per head:
                   score = (x_t M_h) . x_s. Full rank, d^2 parameters per head
                   instead of 2*d*qk_dim. Values still use W_V (d -> head_dim).

Other switches: pos in {learned, rope, none}, mlp in {relu, gelu, swiglu},
init in {old, scaled}, bias on/off, dropout. "old" reproduces the original
notebook (N(0, 0.02) everywhere, biases, lm_head bias).
"""
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class GPTConfig:
    vocab: int = 512
    block: int = 512
    n_layer: int = 4
    n_embd: int = 384
    n_head: int = 6
    head_dim: int = 0      # 0 -> n_embd // n_head
    qk_dim: int = 0        # 0 -> head_dim (mha only)
    attn: str = "mha"      # mha | bilinear
    pos: str = "rope"      # learned | rope | none
    mlp: str = "gelu"      # relu | gelu | swiglu
    init: str = "scaled"   # old | scaled
    bias: bool = False
    dropout: float = 0.0

    def resolved(self):
        c = GPTConfig(**asdict(self))
        if not c.head_dim:
            c.head_dim = c.n_embd // c.n_head
        if not c.qk_dim:
            c.qk_dim = c.head_dim
        return c


def rope_cache(T, dim, base=10000.0, device=None):
    inv = 1.0 / (base ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(T, device=device).float()
    f = torch.outer(t, inv)  # T, dim/2
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    # x: B, H, T, D ; rotate pairs (even, odd)
    T = x.shape[-2]
    c, s = cos[:T].to(x.dtype), sin[:T].to(x.dtype)
    x1, x2 = x[..., 0::2], x[..., 1::2]
    y1 = x1 * c - x2 * s
    y2 = x1 * s + x2 * c
    return torch.stack((y1, y2), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        d, h = c.n_embd, c.n_head
        if c.attn == "mha":
            self.qkv = nn.Linear(d, h * (2 * c.qk_dim + c.head_dim), bias=c.bias)
            self.qk_dim = c.qk_dim
        elif c.attn == "bilinear":
            self.q_bil = nn.Linear(d, h * d, bias=False)   # rows of M_h
            self.v = nn.Linear(d, h * c.head_dim, bias=c.bias)
            self.qk_dim = d
        else:
            raise ValueError(c.attn)
        self.proj = nn.Linear(h * c.head_dim, d, bias=c.bias)
        self.proj.is_residual_out = True
        self.drop = nn.Dropout(c.dropout)
        if c.pos == "rope":
            cos, sin = rope_cache(c.block, self.qk_dim)
            self.register_buffer("cos", cos, persistent=False)
            self.register_buffer("sin", sin, persistent=False)

    def qkv_heads(self, x):
        B, T, d = x.shape
        c, h = self.c, self.c.n_head
        if c.attn == "mha":
            qkv = self.qkv(x)
            q, k, v = qkv.split([h * c.qk_dim, h * c.qk_dim, h * c.head_dim], dim=-1)
            q = q.view(B, T, h, c.qk_dim).transpose(1, 2)
            k = k.view(B, T, h, c.qk_dim).transpose(1, 2)
        else:
            q = self.q_bil(x).view(B, T, h, d).transpose(1, 2)
            k = x.unsqueeze(1).expand(B, h, T, d)
            v = self.v(x)
        v = v.view(B, T, h, c.head_dim).transpose(1, 2)
        if c.pos == "rope":
            q = apply_rope(q, self.cos, self.sin)
            k = apply_rope(k, self.cos, self.sin)
        return q, k, v

    def forward(self, x, head_mask=None, return_attn=False):
        B, T, _ = x.shape
        q, k, v = self.qkv_heads(x)
        if return_attn or head_mask is not None:
            # explicit path, used only for analysis
            att = (q.float() @ k.float().transpose(-2, -1)) / math.sqrt(q.shape[-1])
            mask = torch.ones(T, T, dtype=torch.bool, device=x.device).tril()
            att = att.masked_fill(~mask, float("-inf")).softmax(-1)
            y = att.to(v.dtype) @ v
            if head_mask is not None:
                y = y * head_mask.view(1, -1, 1, 1).to(y.dtype)
        else:
            att = None
            y = F.scaled_dot_product_attention(
                q, k.contiguous(), v, is_causal=True,
                dropout_p=self.c.dropout if self.training else 0.0)
        y = y.transpose(1, 2).reshape(B, T, -1)
        y = self.drop(self.proj(y))
        return (y, att) if return_attn else y


class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c.n_embd
        self.kind = c.mlp
        if c.mlp == "swiglu":
            hdim = int(round(8 * d / 3 / 64)) * 64  # parameter-matched to 4d
            self.w1 = nn.Linear(d, hdim, bias=c.bias)
            self.w3 = nn.Linear(d, hdim, bias=c.bias)
            self.w2 = nn.Linear(hdim, d, bias=c.bias)
            self.w2.is_residual_out = True
        else:
            self.fc = nn.Linear(d, 4 * d, bias=c.bias)
            self.out = nn.Linear(4 * d, d, bias=c.bias)
            self.out.is_residual_out = True
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x):
        if self.kind == "swiglu":
            return self.drop(self.w2(F.silu(self.w1(x)) * self.w3(x)))
        act = F.relu if self.kind == "relu" else F.gelu
        return self.drop(self.out(act(self.fc(x))))


class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c.n_embd, bias=c.bias)
        self.attn = Attention(c)
        self.ln2 = nn.LayerNorm(c.n_embd, bias=c.bias)
        self.mlp = MLP(c)

    def forward(self, x, head_mask=None, return_attn=False):
        if return_attn:
            a, att = self.attn(self.ln1(x), head_mask, True)
            x = x + a
            return x + self.mlp(self.ln2(x)), att
        x = x + self.attn(self.ln1(x), head_mask)
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        c = cfg.resolved()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab, c.n_embd)
        self.pos = nn.Embedding(c.block, c.n_embd) if c.pos == "learned" else None
        self.drop = nn.Dropout(c.dropout)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd, bias=c.bias)
        self.lm_head = nn.Linear(c.n_embd, c.vocab, bias=(c.init == "old"))
        self.lm_head.weight = self.tok.weight  # tied
        self.apply(self._init)
        if c.init == "scaled":
            for m in self.modules():
                if getattr(m, "is_residual_out", False):
                    nn.init.normal_(m.weight, 0.0, 0.02 / math.sqrt(2 * c.n_layer))

    def _init(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def n_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n -= self.tok.weight.numel()
            if self.pos is not None:
                n -= self.pos.weight.numel()
        return n

    def flops_per_token(self):
        c = self.cfg
        n = self.n_params(non_embedding=True) + c.vocab * c.n_embd  # lm_head matmul
        attn = c.n_layer * c.block * c.n_head * (c.qk_dim if c.attn == "mha" else c.n_embd)
        return 6 * n + 6 * attn  # fwd+bwd; attention term averaged over causal window

    def forward(self, idx, targets=None, head_mask=None, return_attn=False,
                reduction="mean"):
        B, T = idx.shape
        x = self.tok(idx)
        if self.pos is not None:
            x = x + self.pos(torch.arange(T, device=idx.device))
        x = self.drop(x)
        atts = []
        for i, blk in enumerate(self.blocks):
            hm = None if head_mask is None else head_mask[i]
            if return_attn:
                x, a = blk(x, hm, True)
                atts.append(a)
            else:
                x = blk(x, hm)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)),
                                   targets.reshape(-1), reduction=reduction)
        if return_attn:
            return logits, loss, atts
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new, temperature=1.0, top_k=None, stop_fn=None):
        for _ in range(max_new):
            ctx = idx[:, -self.cfg.block:]
            logits, _ = self(ctx)
            logits = logits[:, -1, :].float() / temperature
            if top_k:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            nxt = torch.multinomial(logits.softmax(-1), 1)
            idx = torch.cat([idx, nxt], 1)
            if stop_fn is not None and stop_fn(idx):
                break
        return idx


Writing /content/shaaer_code/shaaer/model.py


In [14]:
%%writefile /content/shaaer_code/shaaer/train.py
"""One training run: data on GPU, bf16 autocast, optional torch.compile,
AdamW, warmup + {constant, cosine, wsd} schedule, exact bits-per-character
evaluation on a fixed validation set. Returns a result dict and (optionally)
saves a checkpoint that records its full configuration."""
import json
import math
import os
import time
from dataclasses import asdict

import numpy as np
import torch
import torch.nn.functional as F

from .model import GPT, GPTConfig

LN2 = math.log(2)


def lr_mult(step, total, sched, warmup, decay_frac=0.2, min_ratio=0.1):
    if warmup and step < warmup:
        return (step + 1) / warmup
    if sched == "constant":
        return 1.0
    if sched == "cosine":
        p = (step - warmup) / max(1, total - warmup)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
    if sched in ("wsd", "wsd_sqrt"):
        start = int(total * (1 - decay_frac))
        if step < start:
            return 1.0
        p = (step - start) / max(1, total - start)
        return (1 - p) if sched == "wsd" else (1 - math.sqrt(p))
    raise ValueError(sched)


class Data:
    """Token arrays on the GPU plus a per-token character-length table."""

    def __init__(self, train, val, char_len, device):
        self.train = torch.from_numpy(train.astype(np.int64)).to(device)
        self.val = torch.from_numpy(val.astype(np.int64)).to(device)
        self.char_len = torch.tensor(char_len, dtype=torch.float64, device=device)
        self.device = device

    def batch(self, B, T, gen):
        ix = torch.randint(len(self.train) - T - 1, (B,), device=self.device, generator=gen)
        idx = ix[:, None] + torch.arange(T + 1, device=self.device)
        w = self.train[idx]
        return w[:, :-1].contiguous(), w[:, 1:].contiguous()

    def val_windows(self, T, max_tokens):
        n = min(len(self.val) - 1, max_tokens) // T
        w = self.val[: n * T + 1]
        x = w[:-1].view(n, T)
        y = w[1:].view(n, T)
        return x, y


@torch.no_grad()
def evaluate(model, data, T, max_tokens, batch=64):
    """Exact bits per character over non-overlapping windows of the val split."""
    model.eval()
    X, Y = data.val_windows(T, max_tokens)
    nats, chars = 0.0, 0.0
    for i in range(0, len(X), batch):
        x, y = X[i:i + batch], Y[i:i + batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=x.is_cuda):
            _, loss = model(x, y, reduction="sum")
        nats += loss.item()
        chars += data.char_len[y].sum().item()
    model.train()
    return nats / chars / LN2


def make_optimizer(model, lr, recipe):
    if recipe == "old":
        return torch.optim.AdamW(model.parameters(), lr=lr)  # betas .9/.999, wd .01, all params
    decay, no_decay = [], []
    for p in model.parameters():
        if not p.requires_grad:
            continue
        (decay if p.dim() >= 2 else no_decay).append(p)
    groups = [{"params": decay, "weight_decay": 0.1},
              {"params": no_decay, "weight_decay": 0.0}]
    return torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95), eps=1e-8,
                             fused=torch.cuda.is_available())


def train_run(name, model_cfg: GPTConfig, data, tokens, lr, sched="wsd",
              recipe="new", warmup_frac=0.05, decay_frac=0.2, batch_tokens=131072,
              micro_batch=256, seed=0, eval_every_frac=0.1, eval_tokens=1_000_000,
              final_eval_tokens=8_000_000, compile=True, out_dir=None,
              save_ckpt=False, log=print, extra=None):
    device = data.device
    if compile:
        torch._dynamo.reset()  # each run compiles its own graph; avoid cache-limit fallbacks
    torch.manual_seed(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    T = model_cfg.block
    model = GPT(model_cfg).to(device)
    cfg = model.cfg
    n_params = model.n_params()
    n_nonemb = model.n_params(non_embedding=True)
    fpt = model.flops_per_token()

    micro = min(micro_batch, batch_tokens // T)
    accum = max(1, (batch_tokens // T) // micro)
    B_eff = micro * accum
    steps = max(1, tokens // (B_eff * T))
    warmup = 0 if recipe == "old" else max(50, int(warmup_frac * steps))
    clip = None if recipe == "old" else 1.0

    opt = make_optimizer(model, lr, recipe)
    fwd = torch.compile(model) if (compile and device.startswith("cuda")) else model

    hist = []
    every = max(1, int(steps * eval_every_frac))
    t0 = time.time()
    tok_seen = 0
    run_loss = None
    log(f"[{name}] params {n_params/1e6:.2f}M (non-emb {n_nonemb/1e6:.2f}M) "
        f"steps {steps} x {B_eff*T:,} tok, micro {micro}x{accum}, lr {lr:.2e} {sched}")
    for step in range(steps):
        m = lr_mult(step, steps, sched, warmup, decay_frac)
        for g in opt.param_groups:
            g["lr"] = lr * m
        for _ in range(accum):
            x, y = data.batch(micro, T, gen)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.startswith("cuda")):
                _, loss = fwd(x, y)
            (loss / accum).backward()
        if clip:
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        opt.zero_grad(set_to_none=True)
        tok_seen += B_eff * T
        lv = loss.item() if (step % 50 == 0 or step == steps - 1) else None
        if lv is not None:
            if not math.isfinite(lv):
                log(f"[{name}] diverged at step {step}")
                hist.append(dict(step=step, tokens=tok_seen, train_loss=lv, diverged=True))
                break
            run_loss = lv if run_loss is None else 0.8 * run_loss + 0.2 * lv
        if (step + 1) % every == 0 and step + 1 < steps:
            vb = evaluate(model, data, T, eval_tokens)
            el = time.time() - t0
            hist.append(dict(step=step + 1, tokens=tok_seen, val_bpc=vb,
                             train_loss=run_loss, lr=lr * m, seconds=el))
            log(f"[{name}] {100*(step+1)/steps:4.0f}%  val {vb:.4f} bpc  "
                f"{tok_seen/el/1e6:.2f}M tok/s  {el/60:.1f}m")
    train_seconds = time.time() - t0
    diverged = bool(hist and hist[-1].get("diverged"))
    final = float("nan") if diverged else evaluate(model, data, T, final_eval_tokens)
    hist.append(dict(step=steps, tokens=tok_seen, val_bpc=final, train_loss=run_loss,
                     seconds=train_seconds))
    res = dict(
        name=name, model=asdict(cfg), tokens=tok_seen, lr=lr, sched=sched,
        recipe=recipe, warmup=warmup, decay_frac=decay_frac, batch_tokens=B_eff * T,
        seed=seed, params=n_params, nonemb_params=n_nonemb, flops_per_token=fpt,
        flops=fpt * tok_seen, epochs=tok_seen / len(data.train), final_bpc=final,
        diverged=diverged, seconds=train_seconds, tok_per_s=tok_seen / train_seconds,
        history=hist, extra=extra or {},
    )
    log(f"[{name}] FINAL {final:.4f} bpc  ({train_seconds/60:.1f} min, "
        f"{res['tok_per_s']/1e6:.2f}M tok/s, {res['epochs']:.2f} epochs)")
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        json.dump(res, open(os.path.join(out_dir, f"{name}.json"), "w"), indent=1)
        if save_ckpt:
            torch.save(dict(state_dict=model.state_dict(), cfg=asdict(cfg), result=res),
                       os.path.join(out_dir, f"{name}.pt"))
    del fwd, opt
    return res, model


Overwriting /content/shaaer_code/shaaer/train.py


In [6]:
%%writefile /content/shaaer_code/shaaer/analysis.py
"""What the heads do, and what the model writes.

head_study(model, ...) per (layer, head):
  * ablation: increase in loss when that head's output is zeroed, on all
    tokens and on rhyme tokens only (the last token of each line);
  * attention entropy;
  * rhyme attention: when predicting a line's last token, how much attention
    goes to the last tokens of the poem's previous lines (where the rhyme
    letter lives), relative to uniform attention.

generation_study(model, ...): sample whole poems from [metre|theme] headers and
measure rhyme consistency, repeated rhyme words (ita'), copying, invented words
and hemistich length against the same statistics on real held-out poems.
"""
import math
import random
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from .data import normalize_prompt


def _line_end_positions(x, nl, bad_tokens):
    """For one window x (T,), return list of (q, keys): q predicts the last
    token of a bait line; keys are the last-token positions of previous lines
    of the same poem."""
    T = len(x)
    out = []
    ends = []          # line-final positions in current poem
    for e in range(1, T - 1):
        if x[e + 1] != nl or x[e] == nl:
            if x[e] == nl and e > 0 and x[e - 1] == nl:
                ends = []  # blank line: new poem
            continue
        if x[e] in bad_tokens:     # header line "[...]"
            ends = []
            continue
        q = e - 1
        if ends and q > ends[-1]:
            out.append((q, list(ends)))
        ends.append(e)
    return out


@torch.no_grad()
def head_study(model, data, tok, n_windows=128, batch=16):
    model.eval()
    c = model.cfg
    T = c.block
    nl = tok.stoi["\n"]
    bad = {i for i, t in enumerate(tok.vocab) if "]" in t or "[" in t}
    X, Y = data.val_windows(T, n_windows * T)
    X, Y = X[:n_windows], Y[:n_windows]
    Xc = X.cpu().numpy()
    pairs = [_line_end_positions(Xc[i], nl, bad) for i in range(len(Xc))]
    rhyme_mask = torch.zeros_like(Y, dtype=torch.bool)
    for i, ps in enumerate(pairs):
        for q, _ in ps:
            rhyme_mask[i, q] = True

    L, H = c.n_layer, c.n_head

    def losses(head_mask):
        tot = []
        for i in range(0, len(X), batch):
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=X.is_cuda):
                logits, _ = model(X[i:i + batch], head_mask=head_mask)
            l = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                                Y[i:i + batch].reshape(-1), reduction="none")
            tot.append(l.view(-1, T))
        l = torch.cat(tot)
        return l.mean().item(), l[rhyme_mask].mean().item()

    ones = torch.ones(L, H, device=X.device)
    base_all, base_rhyme = losses(ones)
    abl_all = np.zeros((L, H))
    abl_rhyme = np.zeros((L, H))
    for l in range(L):
        for h in range(H):
            m = ones.clone()
            m[l, h] = 0
            a, r = losses(m)
            abl_all[l, h] = a - base_all
            abl_rhyme[l, h] = r - base_rhyme

    ent = np.zeros((L, H))
    rmass = np.zeros((L, H))
    rbase = 0.0
    n_ent = 0
    n_q = 0
    for i in range(0, len(X), batch):
        xb = X[i:i + batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=X.is_cuda):
            _, _, atts = model(xb, return_attn=True)
        for l, a in enumerate(atts):  # a: B,H,T,T float
            a = a.float()
            p = a[:, :, 64:, :]
            e = -(p.clamp_min(1e-12).log() * p).sum(-1)  # B,H,T-64
            ent[l] += e.mean(dim=(0, 2)).cpu().numpy() * len(xb)
        n_ent += len(xb)
        for j in range(len(xb)):
            for q, keys in pairs[i + j]:
                for l, a in enumerate(atts):
                    rmass[l] += a[j, :, q, keys].float().sum(-1).cpu().numpy()
                rbase += len(keys) / (q + 1)
                n_q += 1
    ent /= max(1, n_ent)
    rmass /= max(1, n_q)
    rbase /= max(1, n_q)
    model.train()
    return dict(
        base_loss=base_all, base_rhyme_loss=base_rhyme,
        ablation_all=abl_all.tolist(), ablation_rhyme=abl_rhyme.tolist(),
        entropy=ent.tolist(), rhyme_mass=rmass.tolist(),
        rhyme_mass_uniform=rbase, n_rhyme_queries=n_q,
    )


# ------------------------------------------------------------------ generation

def _poem_lines(text):
    """Lines of the first poem after the header, complete lines only."""
    body = text.split("\n", 1)[1] if "\n" in text else ""
    if "\n\n" in body:
        body = body.split("\n\n", 1)[0]
        lines = body.split("\n")
    else:
        lines = body.split("\n")[:-1]  # last line may be cut off
    return [l.strip() for l in lines if l.strip()]


def poem_metrics(lines, train_baits=None, train_words=None):
    good = [l for l in lines if l.count(" # ") == 1]
    m = dict(n_lines=len(lines),
             well_formed=len(good) / max(1, len(lines)))
    if len(good) < 2:
        return m
    last_words = [l.split()[-1] for l in good]
    finals = [w[-1] for w in last_words]
    m["rhyme_consistency"] = Counter(finals).most_common(1)[0][1] / len(finals)
    m["repeated_rhyme_word"] = float(len(set(last_words)) < len(last_words))
    if train_baits is not None:
        m["copied"] = sum(hash(l) in train_baits for l in good) / len(good)
    if train_words is not None:
        words = [w for l in good for w in l.replace("#", " ").split()]
        m["invented_words"] = sum(w not in train_words for w in words) / max(1, len(words))
        m["invented_rhyme_words"] = sum(w not in train_words for w in last_words) / len(last_words)
    hl = [len(h.strip()) for l in good for h in l.split(" # ")]
    m["hemistich_chars"] = float(np.mean(hl))
    m["hemistich_chars_sd_within"] = float(np.std(hl))
    return m


def corpus_sets(train_text):
    baits, words = set(), set()
    for line in train_text.split("\n"):
        if " # " in line:
            baits.add(hash(line.strip()))
            words.update(line.replace("#", " ").split())
    return baits, words


def real_poems_by_header(val_text, headers, k=200, seed=0):
    rng = random.Random(seed)
    by = {h: [] for h in headers}
    for block in val_text.split("\n\n"):
        lines = block.strip().split("\n")
        if lines and lines[0] in by:
            by[lines[0]].append(lines[1:])
    return {h: rng.sample(v, min(k, len(v))) for h, v in by.items()}


def _agg(ms):
    keys = sorted({k for m in ms for k in m})
    out = {}
    for k in keys:
        v = [m[k] for m in ms if k in m]
        out[k] = float(np.mean(v)) if v else float("nan")
    out["n_poems"] = len(ms)
    return out


@torch.no_grad()
def generation_study(model, tok, headers, train_sets, n_per_header=48, n_lines=8,
                     temperature=1.0, top_k=20, max_new=400, seed=0):
    model.eval()
    torch.manual_seed(seed)
    dev = next(model.parameters()).device
    nl = tok.stoi["\n"]
    train_baits, train_words = train_sets
    res, samples = {}, {}
    for h in headers:
        p = tok.encode(normalize_prompt(h) + "\n")
        idx = torch.tensor([p] * n_per_header, device=dev)
        plen = len(p)

        def stop(i):
            return bool(((i[:, plen:] == nl).sum(1) >= n_lines + 1).all())

        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
            out = model.generate(idx, max_new, temperature, top_k, stop)
        texts = [tok.decode(r.tolist()) for r in out]
        ms = [poem_metrics(_poem_lines(t)[:n_lines], train_baits, train_words) for t in texts]
        res[h] = _agg(ms)
        samples[h] = texts[:4]
    model.train()
    overall = _agg([dict(v) for v in res.values()])
    return dict(per_header=res, overall=overall, samples=samples,
                settings=dict(temperature=temperature, top_k=top_k, n=n_per_header))


def real_reference(val_text, headers, train_sets, k=200):
    real = real_poems_by_header(val_text, headers, k)
    res = {}
    for h, poems in real.items():
        ms = [poem_metrics(p[:8], *train_sets) for p in poems]
        res[h] = _agg(ms)
    return dict(per_header=res, overall=_agg([dict(v) for v in res.values()]))


Writing /content/shaaer_code/shaaer/analysis.py


In [15]:
%%writefile /content/shaaer_code/shaaer/plots.py
"""Figures and summary.md from whatever results exist (safe to call any time)."""
import json
import math
import os
import re

import numpy as np

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3})


def load_runs(res_dir):
    runs = {}
    for f in os.listdir(res_dir):
        if f.endswith(".json") and not f.startswith("P"):
            try:
                r = json.load(open(os.path.join(res_dir, f)))
                runs[r["name"]] = r
            except Exception:
                pass
    return runs


def ok(r):
    return r is not None and not r.get("diverged") and math.isfinite(r.get("final_bpc", float("nan")))


def curve(r):
    h = [x for x in r["history"] if "val_bpc" in x and math.isfinite(x["val_bpc"])]
    return [x["tokens"] / 1e6 for x in h], [x["val_bpc"] for x in h]


def fig_ladder(runs, fig_dir, bzip2):
    names = ["ladder0_original", "ladder1_adamw_clip", "ladder2_tuned_lr_wsd",
             "ladder3_no_dropout", "ladder4_scaled_init_nobias", "ladder5_gelu", "att_h6_s0"]
    labels = ["original\nrecipe", "+AdamW β2=.95\nwd .1, clip", "+tuned lr\nwarmup+WSD",
              "+no dropout", "+scaled init\nno bias", "+GELU", "+RoPE\n(new base)"]
    pts = [(l, runs[n]) for n, l in zip(names, labels) if ok(runs.get(n))]
    if len(pts) < 2:
        return None
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].bar(range(len(pts)), [r["final_bpc"] for _, r in pts], color="#3b6ea5")
    for i, (_, r) in enumerate(pts):
        ax[0].text(i, r["final_bpc"], f"{r['final_bpc']:.3f}", ha="center", va="bottom", fontsize=8)
    ax[0].set_xticks(range(len(pts)), [l for l, _ in pts], fontsize=7)
    lo = min(r["final_bpc"] for _, r in pts)
    hi = max(r["final_bpc"] for _, r in pts)
    ax[0].set_ylim(lo - 0.15 * (hi - lo + 0.05), hi + 0.15 * (hi - lo + 0.05))
    ax[0].set_ylabel("val bits / char")
    ax[0].set_title("Recipe ladder: one fix at a time (same model size, same tokens)")
    for l, r in pts:
        x, y = curve(r)
        ax[1].plot(x, y, label=l.replace("\n", " "))
    ax[1].set_xlabel("training tokens (M)")
    ax[1].set_ylabel("val bits / char")
    ax[1].set_title("validation curves")
    ax[1].legend(fontsize=7)
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_ladder.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def fig_lr(runs, fig_dir, fit):
    by = {}
    for n, r in runs.items():
        if n.startswith("lr_d") and ok(r):
            d = r["model"]["n_embd"]
            by.setdefault(d, []).append((r["lr"], r["final_bpc"]))
    if not by:
        return None
    fig, ax = plt.subplots(figsize=(6, 4))
    for d in sorted(by):
        pts = sorted(by[d])
        ax.plot([p[0] for p in pts], [p[1] for p in pts], "o-", label=f"width {d}")
        if fit and str(d) in {str(k) for k in fit.get("best", {})}:
            b = fit["best"].get(str(d), fit["best"].get(d))
            ax.axvline(b, ls=":", lw=0.8, color=ax.lines[-1].get_color())
    ax.set_xscale("log")
    ax.set_xlabel("peak learning rate")
    ax.set_ylabel("val bits / char")
    t = "learning-rate sweep"
    if fit:
        t += f"   best lr ∝ width^{fit['slope']:.2f}"
    ax.set_title(t)
    ax.legend()
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_lr.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def _seeds(runs, prefix):
    return [r for n, r in runs.items() if re.fullmatch(re.escape(prefix) + r"_s\d+", n) and ok(r)]


def _best_lr(runs, prefix):
    c = [runs.get(f"{prefix}_s0{suf}") for suf in ("", "_lrx0.5", "_lrx2.0")]
    c = [r for r in c if ok(r)]
    return min(c, key=lambda r: r["final_bpc"]) if c else None


def fig_heads(runs, fig_dir):
    hs = sorted({int(m.group(1)) for n in runs
                 for m in [re.fullmatch(r"att_h(\d+)_s\d+", n)] if m and ok(runs[n])})
    if not hs:
        return None
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    mean, sd = [], []
    for h in hs:
        v = [r["final_bpc"] for r in _seeds(runs, f"att_h{h}")]
        mean.append(np.mean(v))
        sd.append(np.std(v) if len(v) > 1 else 0)
    ax.errorbar(hs, mean, yerr=sd, fmt="o-", capsize=3, color="#3b6ea5",
                label="multi-head, head dim = width / heads (common lr, mean over seeds)")
    for h, m in zip(hs, mean):
        ax.annotate(f"{m:.3f}", (h, m), textcoords="offset points", xytext=(0, 7),
                    ha="center", fontsize=7)
    bl = [(h, _best_lr(runs, f"att_h{h}")) for h in hs]
    bl = [(h, r) for h, r in bl if r]
    if bl:
        ax.plot([h for h, _ in bl], [r["final_bpc"] for _, r in bl], "x--", color="#7f8c8d",
                label="same, best of 0.5x / 1x / 2x lr (seed 0)")
    others = [("att_bil_h1", "bilinear W_QK, 1 head (full rank)", "#c0392b"),
              ("att_bil_h6", "bilinear W_QK, 6 heads (full rank)", "#e67e22"),
              ("att_bil_h2", "bilinear W_QK, 2 heads", "#e67e22"),
              ("att_wideqk_h6", "6 heads, Q/K dim 384 (factored full rank)", "#27ae60"),
              ("att_wideqk_h2", "2 heads, Q/K dim = width", "#27ae60")]
    for pre, lab, col in others:
        v = _seeds(runs, pre)
        if v:
            b = _best_lr(runs, pre)
            extra = f", best lr {b['final_bpc']:.3f}" if b else ""
            ax.axhline(np.mean([r["final_bpc"] for r in v]), ls="--", color=col,
                       label=f"{lab}: {np.mean([r['final_bpc'] for r in v]):.3f}{extra} "
                             f"({v[0]['params']/1e6:.1f}M params)")
    ax.set_xscale("log", base=2)
    ax.set_xticks(hs, [str(h) for h in hs])
    ax.set_xlabel("number of attention heads (width fixed)")
    ax.set_ylabel("val bits / char")
    ax.set_title("Is multi-head attention worth it?")
    ax.legend(fontsize=6.5)
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_heads.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def fig_head_analysis(res_dir, fig_dir):
    f = os.path.join(res_dir, "P4_heads.json")
    if not os.path.exists(f):
        return None
    H = json.load(open(f))
    names = [n for n in H if H[n]]
    if not names:
        return None
    fig, axes = plt.subplots(len(names), 3, figsize=(12, 1.6 + 1.3 * len(names)), squeeze=False)
    for i, n in enumerate(names):
        h = H[n]
        mats = [(np.array(h["ablation_all"]) * 1000, "Δ loss, all tokens (mnats)"),
                (np.array(h["ablation_rhyme"]) * 1000, "Δ loss, line-final tokens (mnats)"),
                (np.array(h["rhyme_mass"]) / max(1e-9, h["rhyme_mass_uniform"]),
                 "attention to previous line ends / uniform")]
        for j, (M, t) in enumerate(mats):
            ax = axes[i, j]
            im = ax.imshow(M, aspect="auto", cmap="magma")
            ax.set_title(f"{n}: {t}", fontsize=7)
            ax.set_ylabel("layer")
            ax.set_xlabel("head")
            ax.set_xticks(range(M.shape[1]))
            ax.set_yticks(range(M.shape[0]))
            ax.tick_params(labelsize=6)
            ax.grid(False)
            if M.shape[1] <= 12:
                for (yy, xx), v in np.ndenumerate(M):
                    ax.text(xx, yy, f"{v:.0f}" if j < 2 else f"{v:.1f}", ha="center",
                            va="center", fontsize=5, color="w" if v < M.mean() else "k")
            plt.colorbar(im, ax=ax, fraction=0.04)
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_head_analysis.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def fig_curves(runs, fig_dir, names, labels, fname, title):
    pts = [(l, runs[n]) for n, l in zip(names, labels) if ok(runs.get(n))]
    if len(pts) < 2:
        return None
    fig, ax = plt.subplots(figsize=(7, 4))
    for l, r in pts:
        x, y = curve(r)
        ax.plot(x, y, label=f"{l}: {r['final_bpc']:.4f}")
    ax.set_xlabel("training tokens (M)")
    ax.set_ylabel("val bits / char")
    ax.set_title(title)
    ax.legend(fontsize=8)
    ys = [r["final_bpc"] for _, r in pts]
    ax.set_ylim(min(ys) - 0.05, min(ys) + 0.6)
    plt.tight_layout()
    p = os.path.join(fig_dir, fname)
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def fig_misc(runs, fig_dir):
    names = [("att_h6_s0", "base (RoPE, GELU, L4 d384)"), ("pos_learned", "learned positions"),
             ("pos_none", "no positions (NoPE)"), ("mlp_relu", "ReLU MLP"),
             ("mlp_swiglu", "SwiGLU MLP"), ("dropout_0.1", "dropout 0.1"),
             ("dw_L2_d512", "2 layers x 512"), ("dw_L8_d256", "8 layers x 256"),
             ("dw_L16_d192", "16 layers x 192"), ("batch_32k", "batch 32k tokens"),
             ("batch_65k", "batch 65k tokens"), ("batch_262k", "batch 262k tokens")]
    pts = [(l, runs[n]) for n, l in names if ok(runs.get(n))]
    if len(pts) < 2:
        return None
    base = pts[0][1]["final_bpc"] if pts[0][0].startswith("base") else None
    fig, ax = plt.subplots(figsize=(8, 0.6 + 0.35 * len(pts)))
    ys = range(len(pts))
    vals = [r["final_bpc"] for _, r in pts]
    ax.barh(ys, vals, color=["#3b6ea5" if i else "#555" for i in range(len(pts))])
    for y, (l, r) in zip(ys, pts):
        ax.text(r["final_bpc"], y, f" {r['final_bpc']:.4f}  ({r['params']/1e6:.1f}M)",
                va="center", fontsize=7)
    ax.set_yticks(list(ys), [l for l, _ in pts], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(min(vals) - 0.05, max(vals) + 0.08)
    if base:
        ax.axvline(base, ls=":", color="k", lw=0.8)
    ax.set_xlabel("val bits / char (lower is better)")
    ax.set_title("Architecture and batch-size ablations (300M tokens each)")
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_misc.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def scaling_fit(runs):
    by = {}
    for n, r in runs.items():
        if n.startswith("sc_") and ok(r):
            by.setdefault(r["extra"]["budget"], []).append(r)
    fits = []
    for C in sorted(by):
        rs = sorted(by[C], key=lambda r: r["params"])
        if len(rs) < 3:
            fits.append(dict(budget=C, n_runs=len(rs), points=[(r["params"], r["tokens"], r["final_bpc"]) for r in rs]))
            continue
        x = np.log([r["params"] for r in rs])
        y = np.array([r["final_bpc"] for r in rs])
        i = int(y.argmin())
        a, b, c = np.polyfit(x, y, 2)
        if a > 0:
            lx = float(np.clip(-b / (2 * a), x.min(), x.max()))
            interior = 0 < i < len(rs) - 1
        else:
            lx, interior = float(x[i]), False
        n_opt = math.exp(lx)
        fits.append(dict(budget=C, n_runs=len(rs), n_opt=n_opt,
                         d_opt=C / (6 * n_opt), bpc_opt=float(np.polyval([a, b, c], lx)),
                         interior_minimum=interior,
                         points=[(r["params"], r["tokens"], r["final_bpc"]) for r in rs]))
    good = [f for f in fits if "n_opt" in f]
    law = None
    if len(good) >= 2:
        lc = np.log([f["budget"] for f in good])
        a_n, _ = np.polyfit(lc, np.log([f["n_opt"] for f in good]), 1)
        a_d, _ = np.polyfit(lc, np.log([f["d_opt"] for f in good]), 1)
        law = dict(n_opt_exponent=float(a_n), d_opt_exponent=float(a_d))
    return dict(per_budget=fits, law=law)


def fig_scaling(runs, fig_dir, res_dir, train_tokens):
    fit = scaling_fit(runs)
    json.dump(fit, open(os.path.join(res_dir, "P6_scaling_fit.json"), "w"), indent=1)
    if not fit["per_budget"]:
        return None
    fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
    cmap = plt.get_cmap("viridis")
    nb = len(fit["per_budget"])
    for k, f in enumerate(fit["per_budget"]):
        col = cmap(k / max(1, nb - 1))
        P = np.array(f["points"])
        ax[0].plot(P[:, 0], P[:, 2], "o-", color=col, label=f"C = {f['budget']:.0e} FLOPs")
        for p in P:
            ax[0].annotate(f"{p[1]/train_tokens:.1f} ep", (p[0], p[2]), fontsize=6,
                           textcoords="offset points", xytext=(0, 5), ha="center")
        if "n_opt" in f:
            ax[0].plot(f["n_opt"], f["bpc_opt"], "*", ms=12, color=col,
                       markeredgecolor="k" if f["interior_minimum"] else "r")
    ax[0].set_xscale("log")
    ax[0].set_xlabel("parameters")
    ax[0].set_ylabel("val bits / char")
    ax[0].set_title("IsoFLOP curves (★ fitted optimum; red edge = optimum at grid edge)")
    ax[0].legend(fontsize=7)
    good = [f for f in fit["per_budget"] if "n_opt" in f]
    if good:
        Cs = [f["budget"] for f in good]
        ax[1].loglog(Cs, [f["n_opt"] for f in good], "o-", label="optimal parameters N*")
        ax[1].loglog(Cs, [f["d_opt"] for f in good], "s-", label="optimal tokens D*")
        ax[1].axhline(train_tokens, ls=":", color="gray", label="one epoch of the corpus")
        ax[1].set_xlabel("compute budget C (FLOPs)")
        law = fit["law"]
        t = "compute-optimal allocation"
        if law:
            t += f"   N* ∝ C^{law['n_opt_exponent']:.2f}, D* ∝ C^{law['d_opt_exponent']:.2f}"
        ax[1].set_title(t, fontsize=8)
        ax[1].legend(fontsize=7)
    plt.tight_layout()
    p = os.path.join(fig_dir, "fig_scaling.png")
    plt.savefig(p, dpi=150)
    plt.close()
    return p


def table(rows, cols):
    out = ["| " + " | ".join(c for c, _ in cols) + " |",
           "|" + "|".join("---" for _ in cols) + "|"]
    for r in rows:
        out.append("| " + " | ".join(f(r) for _, f in cols) + " |")
    return "\n".join(out)


def summary(runs, res_dir, fig_dir, meta):
    L = ["# Shaaer v2 experiments: summary", ""]
    L.append(f"Data: {meta['train_tokens']:,} train tokens / {meta['val_tokens']:,} val tokens "
             f"(BPE-{meta['vocab_size']}), {meta['n_duplicates']:,} duplicate poems removed, "
             f"{100*meta['val_bait_overlap']:.2f}% of val lines also appear verbatim in train.")
    comp = meta.get("compressors_val", {})
    if comp:
        L.append("Compressors on the val text (bits/char): " +
                 ", ".join(f"{k} {v:.3f}" for k, v in comp.items()))
    L.append("")
    cols = [("run", lambda r: r["name"]),
            ("params", lambda r: f"{r['params']/1e6:.2f}M"),
            ("tokens", lambda r: f"{r['tokens']/1e6:.0f}M"),
            ("epochs", lambda r: f"{r['epochs']:.2f}"),
            ("lr", lambda r: f"{r['lr']:.1e}"),
            ("sched", lambda r: r["sched"]),
            ("val bpc", lambda r: f"{r['final_bpc']:.4f}" if ok(r) else "diverged"),
            ("tok/s", lambda r: f"{r['tok_per_s']/1e6:.2f}M"),
            ("min", lambda r: f"{r['seconds']/60:.1f}")]
    groups = [("Learning rate", "lr_"), ("Recipe ladder", "ladder"), ("Attention", "att_"),
              ("Schedules", "sched_"), ("Positions / MLP / dropout", ("pos_", "mlp_", "dropout")),
              ("Depth vs width", "dw_"), ("Batch size", "batch_"), ("Scaling", "sc_")]
    for title, pre in groups:
        rs = sorted([r for n, r in runs.items() if n.startswith(pre)], key=lambda r: r["name"])
        if rs:
            L += [f"## {title}", "", table(rs, cols), ""]
    f = os.path.join(res_dir, "P6_scaling_fit.json")
    if os.path.exists(f):
        fit = json.load(open(f))
        fence = chr(96) * 3
        L += ["## Scaling fit", "", fence, json.dumps(fit, indent=1)[:4000], fence, ""]
    for gname in ("P4_generation.json", "P10_generation_scaled.json"):
        g = os.path.join(res_dir, gname)
        if os.path.exists(g):
            G = json.load(open(g))
            keys = ["well_formed", "rhyme_consistency", "repeated_rhyme_word", "copied",
                    "invented_words", "invented_rhyme_words", "hemistich_chars"]
            rows = [(n, v["overall"]) for n, v in G.items()]
            L += [f"## Generation ({gname})", "",
                  "| model | " + " | ".join(keys) + " |", "|---" * (len(keys) + 1) + "|"]
            for n, o in rows:
                L.append(f"| {n} | " + " | ".join(
                    f"{o.get(k, float('nan')):.3f}" for k in keys) + " |")
            L.append("")
    figs = sorted(x for x in os.listdir(fig_dir) if x.endswith(".png"))
    L += ["## Figures", ""] + [f"![{x}](../figures/{x})" for x in figs]
    open(os.path.join(res_dir, "summary.md"), "w", encoding="utf-8").write("\n".join(L))


def make_all(res_dir, fig_dir, meta, log=print):
    runs = load_runs(res_dir)
    fit = None
    f = os.path.join(res_dir, "P1_lr_fit.json")
    if os.path.exists(f):
        fit = json.load(open(f))
    bz = meta.get("compressors_val", {}).get("bzip2")
    fig_ladder(runs, fig_dir, bz)
    fig_lr(runs, fig_dir, fit)
    fig_heads(runs, fig_dir)
    fig_head_analysis(res_dir, fig_dir)
    fig_curves(runs, fig_dir, ["att_h6_s0", "sched_constant", "sched_cosine", "sched_wsd_sqrt"],
               ["WSD (linear decay, last 20%)", "constant", "cosine to 10%", "WSD (1-sqrt decay)"],
               "fig_schedules.png", "Learning-rate schedules (same model, same tokens)")
    fig_misc(runs, fig_dir)
    fig_scaling(runs, fig_dir, res_dir, meta["train_tokens"])
    summary(runs, res_dir, fig_dir, meta)


Overwriting /content/shaaer_code/shaaer/plots.py


In [16]:
%%writefile /content/shaaer_code/shaaer/suite.py
"""Overnight experiment suite. Resumable: every finished run writes
results/<name>.json and is skipped on restart. A global deadline stops new
runs that would not finish in time.

Phases, in priority order:
  P0 throughput      micro-batch sizes on the GPU
  P1 lr              learning-rate sweep at widths 192/384/768 -> lr(width) fit
  P2 ladder          original recipe, then one fix at a time -> what was holding it back
  P3 attention       number of heads at fixed width; full-rank bilinear W_QK; wide QK
  P4 analysis        head ablation, rhyme attention, generation metrics
  P5 schedules       constant / cosine / WSD / WSD-sqrt
  P6 scaling         IsoFLOP sweep -> compute-optimal model size on this corpus
  P7 misc            positions, MLP, dropout, depth vs width, batch size
  P8 seeds           second seed for the attention study
  P9 scaling+        largest compute budget
  P10 generation     generation metrics for the best scaled models
"""
import json
import math
import os
import time
import traceback

import numpy as np
import torch

from . import analysis, data as D, plots
from .model import GPT, GPTConfig
from .train import Data, train_run


class Suite:
    def __init__(self, root, hours=9.5, quick=False, device=None, ckpt_local=None,
                 log_name="log.txt"):
        self.root = root
        self.quick = quick
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.res_dir = os.path.join(root, "results")
        self.fig_dir = os.path.join(root, "figures")
        self.ck_drive = os.path.join(root, "checkpoints")
        self.ck_local = ckpt_local or self.ck_drive
        for d in (self.res_dir, self.fig_dir, self.ck_drive, self.ck_local):
            os.makedirs(d, exist_ok=True)
        self.logf = open(os.path.join(root, log_name), "a", encoding="utf-8")
        self.t_start = time.time()
        self.deadline = self.t_start + hours * 3600
        self.flops_rate = None   # achieved FLOP/s, updated after each run
        self.micro = 256
        if self.device.startswith("cuda"):
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True

    # ------------------------------------------------------------ utilities
    def log(self, *a):
        s = " ".join(str(x) for x in a)
        stamp = time.strftime("%H:%M:%S")
        print(f"{stamp} {s}", flush=True)
        self.logf.write(f"{stamp} {s}\n")
        self.logf.flush()

    def path(self, name):
        return os.path.join(self.res_dir, f"{name}.json")

    def done(self, name):
        return os.path.exists(self.path(name))

    def load_res(self, name):
        return json.load(open(self.path(name))) if self.done(name) else None

    def time_left(self):
        return self.deadline - time.time()

    def setup_data(self, poems_fn, vocab_size=512):
        ddir = os.path.join(self.root, "data")
        if not os.path.exists(os.path.join(ddir, "meta.json")):
            poems = poems_fn()
            D.build(ddir, poems, vocab_size=vocab_size, log=self.log)
        self.meta, self.tok, train, val = D.load(ddir)
        self.data = Data(train, val, self.tok.token_char_lengths(), self.device)
        self.train_text_path = os.path.join(ddir, "train_poems.txt")
        self.val_text_path = os.path.join(ddir, "val_poems.txt")
        self.log(f"data: {self.meta['train_tokens']:,} train tokens, "
                 f"{self.meta['val_tokens']:,} val tokens, vocab {self.meta['vocab_size']}, "
                 f"duplicates removed {self.meta['n_duplicates']:,}, "
                 f"val bait overlap {100*self.meta['val_bait_overlap']:.2f}%")

    def cfg(self, **kw):
        base = dict(vocab=self.meta["vocab_size"], block=self.block, n_layer=4,
                    n_embd=384, n_head=6, attn="mha", pos="rope", mlp="gelu",
                    init="scaled", bias=False, dropout=0.0)
        if self.quick:
            base.update(n_layer=2, n_embd=64, n_head=2)
        base.update(kw)
        return GPTConfig(**base)

    def run(self, name, cfg, tokens, lr, save=None, **kw):
        """Train unless already done. save in {None, 'drive', 'local'}."""
        if self.done(name):
            return self.load_res(name)
        fpt = GPT(cfg).flops_per_token() if self.flops_rate else None
        if fpt and self.flops_rate:
            est = fpt * tokens / self.flops_rate + 90
            if est > self.time_left():
                self.log(f"[{name}] skipped: needs ~{est/60:.0f} min, "
                         f"{self.time_left()/60:.0f} min left")
                return None
        micro = kw.pop("micro_batch", self.micro)
        if self.quick:
            kw.setdefault("batch_tokens", 2048)
        for attempt in range(4):
            try:
                out = self.ck_drive if save == "drive" else self.ck_local
                res, model = train_run(
                    name, cfg, self.data, tokens, lr, micro_batch=micro,
                    compile=not self.quick and self.device.startswith("cuda"),
                    out_dir=self.res_dir, save_ckpt=False, log=self.log,
                    eval_tokens=200_000 if self.quick else 1_000_000,
                    final_eval_tokens=400_000 if self.quick else 8_000_000, **kw)
                if save:
                    torch.save(dict(state_dict=model.state_dict(), cfg=res["model"]),
                               os.path.join(out, f"{name}.pt"))
                del model
                if self.device.startswith("cuda"):
                    torch.cuda.empty_cache()
                if not res["diverged"] and res["seconds"] > 30:
                    rate = res["flops"] / res["seconds"]
                    self.flops_rate = rate if self.flops_rate is None else \
                        0.5 * self.flops_rate + 0.5 * rate
                return res
            except Exception as e:
                if "out of memory" not in str(e).lower():
                    self.log(f"[{name}] FAILED\n{traceback.format_exc()}")
                    break
                micro //= 2
                self.log(f"[{name}] OOM, retrying with micro-batch {micro}")
                import gc
                gc.collect()
                torch._dynamo.reset()
                torch.cuda.empty_cache()
                if micro < 8:
                    break
        return None

    def load_model(self, name):
        for d in (self.ck_drive, self.ck_local):
            p = os.path.join(d, f"{name}.pt")
            if os.path.exists(p):
                ck = torch.load(p, map_location=self.device, weights_only=False)
                m = GPT(GPTConfig(**ck["cfg"])).to(self.device)
                m.load_state_dict(ck["state_dict"])
                return m
        return None

    def save_json(self, name, obj):
        json.dump(obj, open(os.path.join(self.res_dir, name), "w"),
                  ensure_ascii=False, indent=1)

    def figures(self):
        try:
            plots.make_all(self.res_dir, self.fig_dir, self.meta, log=self.log)
        except Exception:
            self.log("plotting failed\n" + traceback.format_exc())

    # ------------------------------------------------------------ phases
    def p0_throughput(self):
        out = os.path.join(self.res_dir, "P0_throughput.json")
        if os.path.exists(out):
            self.micro = json.load(open(out))["best_micro"]
            return
        if not self.device.startswith("cuda"):
            self.micro = 16
            json.dump(dict(best_micro=16, rows=[]), open(out, "w"))
            return
        from .train import make_optimizer
        rows = []
        cfg = self.cfg()
        for micro in (64, 128, 256, 512):
            try:
                m = GPT(cfg).to(self.device)
                f = torch.compile(m)
                opt = make_optimizer(m, 1e-3, "new")
                g = torch.Generator(device=self.device)
                g.manual_seed(0)
                for i in range(25):
                    if i == 5:
                        torch.cuda.synchronize()
                        t0 = time.time()
                    x, y = self.data.batch(micro, self.block, g)
                    with torch.autocast("cuda", dtype=torch.bfloat16):
                        _, loss = f(x, y)
                    loss.backward()
                    opt.step()
                    opt.zero_grad(set_to_none=True)
                torch.cuda.synchronize()
                tps = 20 * micro * self.block / (time.time() - t0)
                mem = torch.cuda.max_memory_allocated() / 2**30
                rows.append(dict(micro=micro, tok_per_s=tps, mem_gb=mem))
                self.log(f"[P0] micro {micro}: {tps/1e6:.2f}M tok/s, {mem:.1f} GB")
                del m, f, opt
            except torch.cuda.OutOfMemoryError:
                rows.append(dict(micro=micro, oom=True))
                self.log(f"[P0] micro {micro}: OOM")
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
        ok = [r for r in rows if "tok_per_s" in r]
        best = max(ok, key=lambda r: r["tok_per_s"])
        # prefer the smallest micro-batch within 3% of the best
        best = min((r for r in ok if r["tok_per_s"] > 0.97 * best["tok_per_s"]),
                   key=lambda r: r["micro"])
        self.micro = best["micro"]
        fpt = GPT(cfg).flops_per_token()
        self.flops_rate = 0.6 * best["tok_per_s"] * fpt  # conservative until real runs
        json.dump(dict(best_micro=self.micro, rows=rows,
                       gpu=torch.cuda.get_device_name(0)), open(out, "w"), indent=1)

    def p1_lr(self):
        """Sweep lr per width; extend the grid until the best lr is interior."""
        grids = {192: [1e-3, 2e-3, 4e-3], 384: [5e-4, 1e-3, 2e-3], 768: [2.5e-4, 5e-4, 1e-3]}
        if self.quick:
            grids = {64: [1e-3, 3e-3, 1e-2], 128: [1e-3, 3e-3, 1e-2]}
        best = {}
        for d, lrs in grids.items():
            lrs = sorted(lrs)
            pts = []
            for _ in range(6):
                for lr in lrs:
                    self.run(f"lr_d{d}_{lr:.2e}", self.cfg(n_embd=d, n_head=max(1, d // 64)),
                             self.T_lr, lr)
                pts = [(lr, self.load_res(f"lr_d{d}_{lr:.2e}")) for lr in lrs]
                pts = [(lr, r["final_bpc"]) for lr, r in pts if r and not r["diverged"]
                       and math.isfinite(r["final_bpc"])]
                if len(pts) < 2:
                    break
                i = int(np.argmin([p[1] for p in pts]))
                if i == 0 and pts[0][0] == lrs[0]:
                    lrs = [lrs[0] / 2] + lrs
                elif i == len(pts) - 1 and pts[-1][0] == lrs[-1]:
                    lrs = lrs + [lrs[-1] * 2]
                else:
                    break
            if len(pts) < 2:
                continue
            x = np.log([p[0] for p in pts])
            y = np.array([p[1] for p in pts])
            i = int(y.argmin())
            if 0 < i < len(x) - 1:
                a, b, _ = np.polyfit(x[i - 1:i + 2], y[i - 1:i + 2], 2)
                lx = -b / (2 * a) if a > 0 else x[i]
                lx = float(np.clip(lx, x[i - 1], x[i + 1]))
            else:
                lx = float(x[i])
                self.log(f"[P1] width {d}: optimum still at the grid edge")
            best[d] = math.exp(lx)
        if len(best) >= 2:
            ld = np.log(list(best.keys()))
            ll = np.log(list(best.values()))
            slope, icpt = np.polyfit(ld, ll, 1)
        else:
            slope, icpt = -1.0, math.log(1e-3 * 384)
        fit = dict(best=best, slope=float(slope), intercept=float(icpt))
        self.lr_fit = fit
        self.save_json("P1_lr_fit.json", fit)
        self.log(f"[P1] best lr per width {best}; lr(d) ~ d^{slope:.2f}")

    def lr_for(self, d):
        f = self.lr_fit
        return float(np.clip(math.exp(f["intercept"] + f["slope"] * math.log(d)), 1e-4, 2e-2))

    def p2_ladder(self):
        d = 64 if self.quick else 384
        lr = self.lr_for(d)
        steps = [
            ("ladder0_original", dict(pos="learned", mlp="relu", init="old", bias=True, dropout=0.1),
             dict(recipe="old", sched="constant"), 3e-4),
            ("ladder1_adamw_clip", dict(pos="learned", mlp="relu", init="old", bias=True, dropout=0.1),
             dict(recipe="new", sched="constant"), 3e-4),
            ("ladder2_tuned_lr_wsd", dict(pos="learned", mlp="relu", init="old", bias=True, dropout=0.1),
             dict(recipe="new", sched="wsd"), lr),
            ("ladder3_no_dropout", dict(pos="learned", mlp="relu", init="old", bias=True, dropout=0.0),
             dict(recipe="new", sched="wsd"), lr),
            ("ladder4_scaled_init_nobias", dict(pos="learned", mlp="relu"),
             dict(recipe="new", sched="wsd"), lr),
            ("ladder5_gelu", dict(pos="learned"), dict(recipe="new", sched="wsd"), lr),
        ]
        for name, mk, tk, lr_ in steps:
            self.run(name, self.cfg(**mk), self.T_base, lr_,
                     save="drive" if name == "ladder0_original" else None, **tk)
        self.base()   # ladder6 = base (+RoPE)

    def base(self, seed=0):
        d = 64 if self.quick else 384
        return self.run(f"att_h6_s{seed}" if not self.quick else f"att_h2_s{seed}",
                        self.cfg(), self.T_base, self.lr_for(d), seed=seed,
                        save="drive" if seed == 0 else None)

    def attention_variants(self):
        d = 64 if self.quick else 384
        heads = [1, 2] if self.quick else [1, 2, 3, 6, 12, 24]
        v = []
        for h in heads:
            v.append((f"att_h{h}", dict(n_head=h)))
        v.append(("att_bil_h1", dict(attn="bilinear", n_head=1)))
        v.append((f"att_bil_h{2 if self.quick else 6}",
                  dict(attn="bilinear", n_head=2 if self.quick else 6)))
        v.append((f"att_wideqk_h{2 if self.quick else 6}",
                  dict(n_head=2 if self.quick else 6, qk_dim=d)))
        return v

    def p3_attention(self, seed=0):
        d = 64 if self.quick else 384
        for name, mk in self.attention_variants():
            self.run(f"{name}_s{seed}", self.cfg(**mk), self.T_base, self.lr_for(d),
                     seed=seed, save="drive" if seed == 0 else None)

    def p3b_attention_lr(self):
        """Each attention variant at 0.5x and 2x the common lr (seed 0)."""
        d = 64 if self.quick else 384
        for name, mk in self.attention_variants():
            for mult in (0.5, 2.0):
                self.run(f"{name}_s0_lrx{mult}", self.cfg(**mk), self.T_base,
                         self.lr_for(d) * mult)

    def p4_analysis(self):
        out = os.path.join(self.res_dir, "P4_heads.json")
        heads = json.load(open(out)) if os.path.exists(out) else {}
        for name, _ in self.attention_variants():
            key = f"{name}_s0"
            if key in heads:
                continue
            m = self.load_model(key)
            if m is None:
                continue
            try:
                heads[key] = analysis.head_study(m, self.data, self.tok,
                                                 n_windows=16 if self.quick else 128)
                self.log(f"[P4] head study {key} done")
            except Exception:
                self.log(f"[P4] head study {key} failed\n{traceback.format_exc()}")
            del m
            json.dump(heads, open(out, "w"), indent=1)
        self.generation(["ladder0_original", "att_h6_s0" if not self.quick else "att_h2_s0",
                         "att_h1_s0", "att_bil_h1_s0"], "P4_generation.json")

    def _gen_setup(self):
        if getattr(self, "_train_sets", None) is None:
            tt = open(self.train_text_path, encoding="utf-8").read()
            self._train_sets = analysis.corpus_sets(tt)
            del tt
            self._val_text = open(self.val_text_path, encoding="utf-8").read()
            self._headers = [h for h, _ in self.meta["top_headers"][:4]]
        return self._train_sets

    def generation(self, names, fname):
        out = os.path.join(self.res_dir, fname)
        gen = json.load(open(out)) if os.path.exists(out) else {}
        sets = self._gen_setup()
        if "real" not in gen:
            gen["real"] = analysis.real_reference(self._val_text, self._headers, sets)
        for n in names:
            if n in gen:
                continue
            m = self.load_model(n)
            if m is None:
                continue
            try:
                gen[n] = analysis.generation_study(
                    m, self.tok, self._headers, sets,
                    n_per_header=8 if self.quick else 48,
                    max_new=120 if self.quick else 400)
                self.log(f"[gen] {n}: {json.dumps(gen[n]['overall'])}")
            except Exception:
                self.log(f"[gen] {n} failed\n{traceback.format_exc()}")
            del m
            json.dump(gen, open(out, "w"), ensure_ascii=False, indent=1)

    def p5_schedules(self):
        d = 64 if self.quick else 384
        lr = self.lr_for(d)
        for s in ("constant", "cosine", "wsd_sqrt"):
            self.run(f"sched_{s}", self.cfg(), self.T_base, lr, sched=s)

    def scaling_models(self):
        if self.quick:
            return [(1, 32), (2, 64), (2, 96), (3, 128)]
        return [(2, 128), (3, 192), (4, 256), (5, 320), (6, 384), (6, 512),
                (8, 576), (10, 640), (12, 768)]

    def p6_scaling(self, budgets):
        lo, hi = (2e5, 3e6) if self.quick else (64e6, 1.5e9)
        for C in budgets:
            for L, d in self.scaling_models():
                cfg = self.cfg(n_layer=L, n_embd=d, n_head=max(1, d // 64))
                fpt = GPT(cfg).flops_per_token()
                tokens = int(C / fpt)
                if not (lo <= tokens <= hi):
                    continue
                self.run(f"sc_{C:.0e}_L{L}_d{d}", cfg, tokens, self.lr_for(d),
                         save="local", extra=dict(budget=C))
            self.figures()

    def p7_misc(self):
        d = 64 if self.quick else 384
        lr = self.lr_for(d)
        runs = [
            ("pos_learned", dict(pos="learned"), {}),
            ("pos_none", dict(pos="none"), {}),
            ("mlp_relu", dict(mlp="relu"), {}),
            ("mlp_swiglu", dict(mlp="swiglu"), {}),
            ("dropout_0.1", dict(dropout=0.1), {}),
        ]
        if not self.quick:
            runs += [
                ("dw_L2_d512", dict(n_layer=2, n_embd=512, n_head=8), {}),
                ("dw_L8_d256", dict(n_layer=8, n_embd=256, n_head=4), {}),
                ("dw_L16_d192", dict(n_layer=16, n_embd=192, n_head=3), {}),
                ("batch_32k", {}, dict(batch_tokens=32768)),
                ("batch_65k", {}, dict(batch_tokens=65536)),
                ("batch_262k", {}, dict(batch_tokens=262144)),
            ]
        for name, mk, tk in runs:
            dd = mk.get("n_embd", d)
            self.run(name, self.cfg(**mk), self.T_base, self.lr_for(dd), **tk)

    def p10_best_scaled(self):
        rows = [json.load(open(os.path.join(self.res_dir, f)))
                for f in os.listdir(self.res_dir) if f.startswith("sc_") and f.endswith(".json")]
        best = {}
        for r in rows:
            C = r["extra"]["budget"]
            if C not in best or r["final_bpc"] < best[C]["final_bpc"]:
                best[C] = r
        names = [best[C]["name"] for C in sorted(best)]
        for n in names:   # keep the winners on Drive
            src = os.path.join(self.ck_local, f"{n}.pt")
            dst = os.path.join(self.ck_drive, f"{n}.pt")
            if os.path.exists(src) and not os.path.exists(dst):
                import shutil
                shutil.copy(src, dst)
        self.generation(names, "P10_generation_scaled.json")

    # ------------------------------------------------------------ main
    def main(self, poems_fn):
        self.block = 64 if self.quick else 512
        self.T_base = 300_000 if self.quick else 300_000_000
        self.T_lr = 150_000 if self.quick else 200_000_000
        self.setup_data(poems_fn)
        if self.quick:
            self.micro = 16
        phases = [
            ("P0", self.p0_throughput),
            ("P1", self.p1_lr),
            ("P2", self.p2_ladder),
            ("P3", lambda: self.p3_attention(0)),
            ("P3b", self.p3b_attention_lr),
            ("P4", self.p4_analysis),
            ("P5", self.p5_schedules),
            ("P6", lambda: self.p6_scaling([3e9, 1e10] if self.quick else [3e15, 1e16, 3e16, 1e17])),
            ("P7", self.p7_misc),
            ("P8", lambda: self.p3_attention(1)),
            ("P9", lambda: self.p6_scaling([3e10] if self.quick else [3e17])),
            ("P10", self.p10_best_scaled),
        ]
        for tag, fn in phases:
            if self.time_left() < 120:
                self.log(f"deadline reached before {tag}")
                break
            self.log(f"===== {tag}  ({self.time_left()/3600:.2f} h left)")
            try:
                fn()
            except Exception:
                self.log(f"{tag} FAILED\n{traceback.format_exc()}")
            self.figures()
        self.log(f"suite finished in {(time.time()-self.t_start)/3600:.2f} h")


def ashaar_poems():
    import datasets
    ds = datasets.load_dataset("arbml/ashaar", split="train")
    poems, stats = D.poems_from_ashaar(ds)
    print(stats)
    return poems


Overwriting /content/shaaer_code/shaaer/suite.py


In [19]:
%%writefile /content/shaaer_code/shaaer/extras.py
"""Follow-up analyses, run after the main suite on its saved checkpoints.

E1  level-k conditionals: for contexts of k previous tokens that occur often in
    held-out text, compare the model's implied P(next | context) (its full-context
    predictions averaged over every occurrence of the context) with the empirical
    P(next | context). Model predictions are averaged on one half of the val
    windows and compared with counts from the other half; the real-vs-real floor
    compares counts from the two halves. Same pass: loss vs position in the
    window, and calibration (top-1 confidence vs accuracy).
E2  distillation: the 7M base model trained on a bigger teacher's full predicted
    distributions instead of one-hot next tokens (pure, and 50/50 with data).
E3  bilinear W_QK vs standard attention at widths 192 and 768.
E4  generation metrics at temperature 0.7 (robustness of the T=1 results).
"""
import json
import math
import os
import time
import traceback
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn.functional as F

from . import analysis, plots
from .model import GPT, GPTConfig
from .train import evaluate, lr_mult, make_optimizer

LN2 = math.log(2)


def _js(p, q):
    m = 0.5 * (p + q)

    def kl(a, b):
        mask = a > 0
        return float((a[mask] * np.log2(a[mask] / np.maximum(b[mask], 1e-12))).sum())
    return 0.5 * kl(p, m) + 0.5 * kl(q, m)


@torch.no_grad()
def level_k_study(model, data, ks=(0, 1, 2, 3, 4), min_count=60, max_ctx=20000, batch=32):
    model.eval()
    T = model.cfg.block
    V = model.cfg.vocab
    X, Y = data.val_windows(T, 10**9)
    Xn, Yn = X.cpu().numpy(), Y.cpu().numpy()
    n = len(Xn)
    half = np.arange(n) % 2  # window parity: A = 0, B = 1

    def ctx_keys(k):
        # key of the k tokens before target position t is x[t-k+1..t]; valid for t >= k-1
        if k == 0:
            return np.zeros_like(Xn, dtype=np.int64), np.ones_like(Xn, dtype=bool)
        key = np.zeros_like(Xn, dtype=np.int64)
        valid = np.zeros_like(Xn, dtype=bool)
        valid[:, k - 1:] = True
        for j in range(k):
            sh = np.zeros_like(Xn)
            sh[:, k - 1:] = Xn[:, k - 1 - j: T - j]
            key = key * V + sh
        return key, valid

    levels = {}
    for k in ks:
        key, valid = ctx_keys(k)
        uk, uc = np.unique(key[valid], return_counts=True)
        order = np.argsort(-uc)[:max_ctx]
        keep = np.sort(uk[order][uc[order] >= min_count])
        ci = np.full(key.shape, -1, dtype=np.int64)
        if len(keep):
            pos = np.searchsorted(keep, key)
            posc = np.minimum(pos, len(keep) - 1)
            hit = valid & (keep[posc] == key)
            ci[hit] = posc[hit]
        emp = np.zeros((2, len(keep), V))
        for h in (0, 1):
            m = (ci >= 0) & (half[:, None] == h)
            np.add.at(emp[h], (ci[m], Yn[m]), 1)
        levels[k] = dict(ci=torch.from_numpy(ci).to(X.device), emp=emp, n_ctx=len(keep),
                         qsum=torch.zeros(2, len(keep), V, dtype=torch.float64, device=X.device))

    pos_loss = torch.zeros(T, dtype=torch.float64, device=X.device)
    nb = 10
    conf_sum = torch.zeros(nb, dtype=torch.float64, device=X.device)
    acc_sum = torch.zeros(nb, dtype=torch.float64, device=X.device)
    cnt_bin = torch.zeros(nb, dtype=torch.float64, device=X.device)
    half_t = torch.from_numpy(half).to(X.device)
    for i in range(0, n, batch):
        x, y = X[i:i + batch], Y[i:i + batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=x.is_cuda):
            logits, _ = model(x)
        q = logits.float().softmax(-1)  # B,T,V
        lp = torch.log(q.gather(-1, y[..., None]).squeeze(-1).clamp_min(1e-12))
        pos_loss += (-lp).double().sum(0)
        conf, pred = q.max(-1)
        b = (conf * nb).clamp(max=nb - 1e-6).long()
        conf_sum.index_add_(0, b.reshape(-1), conf.reshape(-1).double())
        acc_sum.index_add_(0, b.reshape(-1), (pred == y).reshape(-1).double())
        cnt_bin.index_add_(0, b.reshape(-1), torch.ones_like(conf).reshape(-1).double())
        hb = half_t[i:i + batch]
        for k, L in levels.items():
            if not L["n_ctx"]:
                continue
            c = L["ci"][i:i + batch]
            for h in (0, 1):
                m = (c >= 0) & (hb[:, None] == h)
                if m.any():
                    L["qsum"][h].index_add_(0, c[m], q[m].double())
    out = {"levels": {}}
    for k, L in levels.items():
        if not L["n_ctx"]:
            continue
        qs = L["qsum"].cpu().numpy()
        emp = L["emp"]
        model_js, floor_js, w_tot, cover = 0.0, 0.0, 0.0, 0.0
        for j in range(L["n_ctx"]):
            nA, nB = emp[0, j].sum(), emp[1, j].sum()
            if nA < min_count / 4 or nB < min_count / 4:
                continue
            pA, pB = emp[0, j] / nA, emp[1, j] / nB
            qA = qs[0, j] / qs[0, j].sum()
            qB = qs[1, j] / qs[1, j].sum()
            w = nA + nB
            model_js += w * 0.5 * (_js(qA, pB) + _js(qB, pA))
            floor_js += w * _js(pA, pB)
            w_tot += w
        if w_tot:
            out["levels"][str(k)] = dict(model_js=model_js / w_tot, floor_js=floor_js / w_tot,
                                         n_contexts=L["n_ctx"], coverage=w_tot / Yn.size)
    tokens_per_pos = n
    out["loss_by_position_bits"] = (pos_loss / tokens_per_pos / LN2).cpu().tolist()
    cb = cnt_bin.clamp_min(1)
    out["calibration"] = dict(confidence=(conf_sum / cb).cpu().tolist(),
                              accuracy=(acc_sum / cb).cpu().tolist(),
                              count=cnt_bin.cpu().tolist())
    tot = cnt_bin.sum()
    out["ece"] = float(((conf_sum - acc_sum).abs().sum() / tot).item())
    model.train()
    return out


def distill_run(name, student_cfg, teacher, data, tokens, lr, alpha=1.0, batch_tokens=131072,
                micro=128, seed=0, out_dir=None, log=print, compile=True):
    """alpha=1: pure soft targets from the teacher; alpha=0.5: half data, half teacher."""
    device = data.device
    if compile:
        torch._dynamo.reset()
    torch.manual_seed(seed)
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    T = student_cfg.block
    model = GPT(student_cfg).to(device)
    teacher.eval()
    accum = max(1, (batch_tokens // T) // micro)
    steps = max(1, tokens // (micro * accum * T))
    warmup = max(50, int(0.05 * steps))
    opt = make_optimizer(model, lr, "new")
    fwd = torch.compile(model) if (compile and device.startswith("cuda")) else model
    tfwd = torch.compile(teacher) if (compile and device.startswith("cuda")) else teacher
    t0 = time.time()
    hist = []
    every = max(1, steps // 10)
    log(f"[{name}] distillation alpha={alpha}, {steps} steps, lr {lr:.2e}")
    for step in range(steps):
        m = lr_mult(step, steps, "wsd", warmup)
        for g in opt.param_groups:
            g["lr"] = lr * m
        for _ in range(accum):
            x, y = data.batch(micro, T, gen)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=device.startswith("cuda")):
                with torch.no_grad():
                    tl, _ = tfwd(x)
                sl, _ = fwd(x)
            logp = F.log_softmax(sl.float(), -1)
            soft = -(F.softmax(tl.float(), -1) * logp).sum(-1).mean()
            hard = F.nll_loss(logp.view(-1, logp.size(-1)), y.reshape(-1))
            loss = alpha * soft + (1 - alpha) * hard
            (loss / accum).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        opt.zero_grad(set_to_none=True)
        if (step + 1) % every == 0 and step + 1 < steps:
            vb = evaluate(model, data, T, 1_000_000)
            hist.append(dict(step=step + 1, tokens=(step + 1) * micro * accum * T, val_bpc=vb,
                             seconds=time.time() - t0))
            log(f"[{name}] {100*(step+1)/steps:4.0f}%  val {vb:.4f} bpc")
    final = evaluate(model, data, T, 8_000_000)
    tok_seen = steps * micro * accum * T
    hist.append(dict(step=steps, tokens=tok_seen, val_bpc=final, seconds=time.time() - t0))
    res = dict(name=name, model=student_cfg.__dict__ | {}, tokens=tok_seen, lr=lr, sched="wsd",
               recipe="distill", alpha=alpha, seed=seed, params=model.n_params(),
               nonemb_params=model.n_params(True), final_bpc=final, diverged=False,
               seconds=time.time() - t0, tok_per_s=tok_seen / (time.time() - t0),
               epochs=tok_seen / len(data.train), history=hist, flops=0,
               flops_per_token=0, batch_tokens=micro * accum * T, extra={})
    log(f"[{name}] FINAL {final:.4f} bpc")
    if out_dir:
        json.dump(res, open(os.path.join(out_dir, f"{name}.json"), "w"), indent=1, default=str)
    return res


def run_extras(S, hours=1.5):
    """S: a Suite already set up (data loaded, lr fit available)."""
    t_end = time.time() + hours * 3600
    out = os.path.join(S.res_dir, "E_results.json")
    E = json.load(open(out)) if os.path.exists(out) else {}

    def save():
        json.dump(E, open(out, "w"), ensure_ascii=False, indent=1)

    runs = plots.load_runs(S.res_dir)

    # E3 bilinear vs standard at other widths (cheap at 192, ~7 min each at 768)
    for d in ((64,) if S.quick else (192, 768)):
        for name, mk in ((f"e3_mha_d{d}", dict(n_embd=d, n_head=max(1, d // 64))),
                         (f"e3_bil1_d{d}", dict(n_embd=d, n_head=1, attn="bilinear"))):
            if time.time() > t_end:
                break
            S.run(name, S.cfg(**mk), S.T_base, S.lr_for(d))

    # teacher = lowest-bpc model with a checkpoint
    cands = []
    for d in (S.ck_drive, S.ck_local):
        if os.path.isdir(d):
            for f in os.listdir(d):
                if f.endswith(".pt"):
                    r = runs.get(f[:-3])
                    if r and plots.ok(r):
                        cands.append((r["final_bpc"], f[:-3]))
    cands.sort()
    teacher_name = cands[0][1] if cands else None
    base_name = "att_h2_s0" if S.quick else "att_h6_s0"

    # E1 level-k conditionals for: original recipe, base, 1-head, bilinear 1-head, teacher
    E.setdefault("levels", {})
    for n in ["ladder0_original", base_name, "att_h1_s0", "att_bil_h1_s0", teacher_name]:
        if not n or n in E["levels"] or time.time() > t_end:
            continue
        m = S.load_model(n)
        if m is None:
            continue
        try:
            E["levels"][n] = level_k_study(m, S.data, min_count=10 if S.quick else 60)
            S.log(f"[E1] {n}: " + ", ".join(
                f"k={k} model {v['model_js']:.4f} / floor {v['floor_js']:.4f}"
                for k, v in E["levels"][n]["levels"].items()) + f", ECE {E['levels'][n]['ece']:.4f}")
        except Exception:
            S.log(f"[E1] {n} failed\n{traceback.format_exc()}")
        del m
        save()

    # E2 distillation into the base configuration
    if teacher_name and time.time() < t_end:
        teacher = S.load_model(teacher_name)
        base = runs.get(base_name)
        if teacher is not None and base is not None:
            for alpha in (1.0, 0.5):
                name = f"e2_distill_a{alpha}"
                if S.done(name) or time.time() > t_end:
                    continue
                try:
                    distill_run(name, S.cfg(), teacher, S.data, S.T_base, base["lr"], alpha=alpha,
                                micro=S.micro if S.micro <= 128 else 128, out_dir=S.res_dir,
                                log=S.log, compile=not S.quick and S.device.startswith("cuda"))
                except Exception:
                    S.log(f"[E2] {name} failed\n{traceback.format_exc()}")
            E["distill_teacher"] = teacher_name
            save()
            del teacher

    # E4 generation at temperature 0.7
    E.setdefault("gen_t07", {})
    sets = S._gen_setup()
    for n in [base_name, teacher_name]:
        if not n or n in E["gen_t07"] or time.time() > t_end:
            continue
        m = S.load_model(n)
        if m is None:
            continue
        E["gen_t07"][n] = analysis.generation_study(m, S.tok, S._headers, sets,
                                                    n_per_header=8 if S.quick else 48,
                                                    temperature=0.7,
                                                    max_new=120 if S.quick else 400)
        del m
        save()
    E["real_reference"] = analysis.real_reference(S._val_text, S._headers, sets)
    save()
    S.figures()
    try:
        fig_levels(E, S.fig_dir)
    except Exception:
        S.log("level figure failed\n" + traceback.format_exc())
    S.log("extras finished")


def fig_levels(E, fig_dir):
    import matplotlib.pyplot as plt
    L = E.get("levels", {})
    if not L:
        return
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    floor_done = False
    for n, r in L.items():
        ks = sorted(r["levels"], key=int)
        ax[0].plot([int(k) for k in ks], [r["levels"][k]["model_js"] for k in ks], "o-", label=n)
        if not floor_done:
            ax[0].plot([int(k) for k in ks], [r["levels"][k]["floor_js"] for k in ks], "k--",
                       label="real vs real (sampling floor)")
            floor_done = True
        lp = np.array(r["loss_by_position_bits"])
        ax[1].plot(np.arange(len(lp)), lp, lw=0.8, label=n)
        c = r["calibration"]
        ax[2].plot(c["confidence"], c["accuracy"], "o-", ms=3, label=f"{n} (ECE {r['ece']:.3f})")
    ax[0].set_xlabel("k = tokens of context")
    ax[0].set_ylabel("Jensen-Shannon divergence (bits)")
    ax[0].set_title("Model vs empirical P(next | last k tokens)")
    ax[0].legend(fontsize=6)
    ax[1].set_xlabel("position in the 512-token window")
    ax[1].set_ylabel("loss (bits / token)")
    ax[1].set_title("How much context helps")
    ax[1].set_xscale("log")
    ax[1].legend(fontsize=6)
    ax[2].plot([0, 1], [0, 1], "k:", lw=0.8)
    ax[2].set_xlabel("model confidence (top-1 probability)")
    ax[2].set_ylabel("fraction correct")
    ax[2].set_title("Calibration")
    ax[2].legend(fontsize=6)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, "fig_levels.png"), dpi=150)
    plt.close()


def main(root, hours=1.5, quick=False, poems_fn=None):
    from .suite import Suite
    S = Suite(root, hours=hours, quick=quick, ckpt_local=None if quick else "/content/ckpt",
              log_name="log.txt")
    S.block = 64 if quick else 512
    S.T_base = 300_000 if quick else 300_000_000
    S.T_lr = 150_000 if quick else 200_000_000
    S.setup_data(poems_fn)
    S.p0_throughput()
    fit = json.load(open(os.path.join(S.res_dir, "P1_lr_fit.json")))
    fit["best"] = {int(k): v for k, v in fit["best"].items()}
    S.lr_fit = fit
    S.log("===== extras")
    run_extras(S, hours)


Writing /content/shaaer_code/shaaer/extras.py


In [17]:
# 2. Check the code arrived intact, and that every attention variant runs on this GPU
import ast, hashlib, sys, torch
sys.path.insert(0, '/content/shaaer_code')
expected = dict(tok='991c3ddac1', data='e3657e9575', model='79d059be18', train='36932b28ba',
                analysis='abf2c32f72', plots='ee5b49e59f', suite='243180d985')
for f, h in expected.items():
    src = open(f'/content/shaaer_code/shaaer/{f}.py', encoding='utf-8').read()
    got = hashlib.md5(ast.dump(ast.parse(src)).encode()).hexdigest()[:10]
    print(f'{f:9s}', 'OK' if got == h else f'MISMATCH {got}')
from shaaer.model import GPT, GPTConfig
for kw in [dict(), dict(n_head=1), dict(n_head=24), dict(attn='bilinear', n_head=1),
           dict(attn='bilinear', n_head=6), dict(qk_dim=384), dict(pos='learned', mlp='relu', init='old', bias=True, dropout=0.1),
           dict(pos='none', mlp='swiglu')]:
    m = GPT(GPTConfig(**kw)).cuda()
    x = torch.randint(0, 512, (8, 512), device='cuda')
    with torch.autocast('cuda', dtype=torch.bfloat16):
        _, loss = m(x, x)
    loss.backward()
    print(kw, f'params {m.n_params()/1e6:.2f}M  loss {loss.item():.3f}')
print(torch.__version__, torch.cuda.get_device_name(0))

tok       OK
data      OK
model     OK
train     OK
analysis  OK
plots     OK
suite     OK
{} params 7.28M  loss 5.160
{'n_head': 1} params 7.28M  loss 5.143
{'n_head': 24} params 7.28M  loss 5.139
{'attn': 'bilinear', 'n_head': 1} params 6.69M  loss 5.106
{'attn': 'bilinear', 'n_head': 6} params 9.64M  loss 5.177
{'qk_dim': 384} params 13.18M  loss 5.123
{'pos': 'learned', 'mlp': 'relu', 'init': 'old', 'bias': True, 'dropout': 0.1} params 7.49M  loss 6.024
{'pos': 'none', 'mlp': 'swiglu'} params 7.28M  loss 3.926
2.14.1+cu130 NVIDIA A100-SXM4-80GB
cleared: ['log.txt', 'data']


In [18]:
# 3. Run the overnight suite (~9.5 h on an A100). Resumable: if the runtime disconnects,
#    run cells 1-3 again and finished runs are skipped. Progress: MyDrive/shaaer_v2/log.txt,
#    figures: MyDrive/shaaer_v2/figures/, tables: MyDrive/shaaer_v2/results/summary.md
!pip -q install datasets
!cd /content/shaaer_code && python -u -c "from shaaer.suite import Suite, ashaar_poems; Suite('/content/drive/MyDrive/shaaer_v2', hours=9.5, ckpt_local='/content/ckpt').main(ashaar_poems)"

21:06:13 data: 96,356,960 train tokens, 3,795,905 val tokens, vocab 512, duplicates removed 3,138, val bait overlap 0.00%
21:06:13 ===== P0  (9.50 h left)
21:06:19 [P0] micro 64: 2.10M tok/s, 3.0 GB
21:06:21 [P0] micro 128: 2.31M tok/s, 5.2 GB
21:06:23 [P0] micro 256: 2.34M tok/s, 9.4 GB
21:06:26 [P0] micro 512: 2.37M tok/s, 18.0 GB
21:06:26 ===== P1  (9.50 h left)
21:06:26 [lr_d192_1.00e-03] params 1.87M (non-emb 1.77M) steps 1525 x 131,072 tok, micro 128x2, lr 1.00e-03 wsd
21:06:31 [lr_d192_1.00e-03]   10%  val 3.1089 bpc  3.86M tok/s  0.1m
21:06:35 [lr_d192_1.00e-03]   20%  val 2.8480 bpc  4.23M tok/s  0.2m
21:06:40 [lr_d192_1.00e-03]   30%  val 2.7083 bpc  4.37M tok/s  0.2m
21:06:44 [lr_d192_1.00e-03]   40%  val 2.6130 bpc  4.44M tok/s  0.3m
21:06:48 [lr_d192_1.00e-03]   50%  val 2.5566 bpc  4.49M tok/s  0.4m
21:06:52 [lr_d192_1.00e-03]   60%  val 2.5070 bpc  4.51M tok/s  0.4m
21:06:57 [lr_d192_1.00e-03]   70%  val 2.4657 bpc  4.53M tok/s  0.5m
21:07:01 [lr_d192_1.00e-03]   80%  va

In [20]:
# 4. Follow-ups on the saved models (~1.5 h): level-k conditionals vs empirical, calibration,
#    loss by context length, distillation from the best model, bilinear vs standard at widths
#    192/768, generation at temperature 0.7. Results: results/E_results.json, figures/fig_levels.png
import ast, hashlib
src = open('/content/shaaer_code/shaaer/extras.py', encoding='utf-8').read()
print('extras', 'OK' if hashlib.md5(ast.dump(ast.parse(src)).encode()).hexdigest()[:10] == 'b7206ab420' else 'MISMATCH')
!cd /content/shaaer_code && python -u -c "from shaaer.extras import main; main('/content/drive/MyDrive/shaaer_v2', hours=1.5)"

extras OK
04:08:19 data: 96,356,960 train tokens, 3,795,905 val tokens, vocab 512, duplicates removed 3,138, val bait overlap 0.00%
04:08:19 ===== extras
04:08:22 [e3_mha_d192] params 1.87M (non-emb 1.77M) steps 2288 x 131,072 tok, micro 128x2, lr 2.93e-03 wsd
04:08:32 [e3_mha_d192]   10%  val 2.8764 bpc  3.02M tok/s  0.2m
04:08:38 [e3_mha_d192]   20%  val 2.6088 bpc  3.69M tok/s  0.3m
04:08:44 [e3_mha_d192]   30%  val 2.4917 bpc  3.98M tok/s  0.4m
04:08:51 [e3_mha_d192]   40%  val 2.4150 bpc  4.14M tok/s  0.5m
04:08:57 [e3_mha_d192]   50%  val 2.3646 bpc  4.24M tok/s  0.6m
04:09:04 [e3_mha_d192]   60%  val 2.3308 bpc  4.31M tok/s  0.7m
04:09:10 [e3_mha_d192]   70%  val 2.3033 bpc  4.36M tok/s  0.8m
04:09:16 [e3_mha_d192]   80%  val 2.2814 bpc  4.40M tok/s  0.9m
04:09:23 [e3_mha_d192]   90%  val 2.2405 bpc  4.43M tok/s  1.0m
04:09:29 [e3_mha_d192]  100%  val 2.2036 bpc  4.45M tok/s  1.1m
04:09:30 [e3_mha_d192] FINAL 2.2079 bpc  (1.1 min, 4.45M tok/s, 3.11 epochs)
04:09:30 [e3_bil1_d192

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import json, glob
R='/content/drive/MyDrive/shaaer_v2/results/'
for f in ['P10_generation_scaled.json','E_results.json','P4_generation.json']:
    try:
        d=json.load(open(R+f))
    except Exception as e:
        print(f, e); continue
    def walk(o, path=''):
        if isinstance(o, dict):
            for k,v in o.items():
                if k=='samples' and isinstance(v, dict):
                    for h,txts in v.items():
                        print('=====', f, path, h)
                        for t in txts[:2]: print(t); print('---')
                else: walk(v, path+'/'+str(k))
    walk(d)

Mounted at /content/drive
===== P10_generation_scaled.json /sc_3e+15_L4_d256 [غير معروف|عامه]
[غير معروف|عامه]
عجبت لقلبي ان لاحمي # واقصر لي مذ تستوي
وان تمت من عروتهم # كعبا ان تؤذيهم القضيب
لانك ماجد بالجدي # وان سمعت علي الصدي القضيب
وانكسار النور في حفمي # وانكسار الخوف لا تستضيء
وانت في رحله الحسن # وانت في الشوق لم تزل تستحي
فمزق بالاسي مغاني # يداعبه علي شفتها
وتدق حما للعوادي # وتنزل للحب من غصنا

[غير معروف|عامه]
حيرتنا ماذا من حياه # ان تتبعنا الاخري
يا قبر لا تضوع اليها # ولا تعجبها ولا تشري المني
ما في العيون من التعجيل # لا ينتشيها ولا تردح
ما للهوي ما دمت تتوقيع # والله في القلوب ما سمعت بهن

[الخفيف|عامه]
يا صديقي من سماح الاسي عالي # انها للعالم الا الماثق البهالا
يا اخا الحسين لشفاء الاسي #
---
[غير معروف|عامه]
ماذا تري قد رقدت حول نوار # من صداها كاد ينهض الاقدار
قد عجزت الله عصر النصر يحيي له # من ان يري لي من حياه الدهر عذار
قد جاء في حماه الله واحتمل # ما كان في قرب عصر ولا تفار
هذا لعينيك لا تدري نسي له # يوما ولا ترجيه مما تقاري
والله ما يفديك الا لانه # ما كان 